# S6E10 Step by Step: <TODO: topics> — Every Concept Explained

<!-- DÍA 0: esqueleto sacado de S6E9. Las secciones se quedan; la narrativa se reescribe con los datos de S6E10.
     Todo lo marcado TODO se rellena el día 1. Borrar este comentario antes de publicar. -->

This notebook solves the October 2026 Playground competition (**<TODO: what we predict>**)
from scratch, and **explains every concept the first time it appears**: what it is, why we need it,
and what it tells us about *this* dataset.

It is written for anyone who wants to *understand* a complete tabular ML workflow, not just copy a score.
If you already know a concept, skip its box.

**What you will find**

| Part | Topics |
|---|---|
| 1. The problem and the data | task type, variable types, target distribution |
| 2. The metric | <TODO: metric> — what it rewards and what it ignores |
| 3. EDA | what drives the target |
| 4. Validation before modelling | overfitting, K-fold, out-of-fold predictions, CV vs leaderboard |
| 5. Model 1: <TODO: linear baseline> | |
| 6. Model 2: LightGBM | gradient boosting, early stopping |
| 7. Feature engineering | <TODO> |
| 8. Interpretation with SHAP | |
| 9. What helped and what did not | an honest ablation table |
| 10. Summary | key takeaways and a glossary |

**Results at a glance** (<TODO>-fold CV; public leaderboard where submitted)

| Model | CV <TODO: metric> | Public LB |
|---|---|---|
| <TODO> | | |

Every number quoted in the text is produced by the code cells of this notebook.

# Part 1 — The problem and the data

## 1.1 What are we predicting?

- **One row = <TODO>**.
- **Target**: `<TODO>` — <TODO>.
- **Task type**: <TODO: binary classification / regression / multiclass>.

The data is **synthetic**: Kaggle generated it from the
[<TODO: original dataset>](<TODO: URL>) dataset. Keep this in mind:
the patterns we find are the ones the generator put there — and they can be exploited.

In [ ]:
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)

SEED = 42
N_FOLDS = 5
TARGET = "TODO"  # día 1: nombre real de la columna objetivo

# Works on Kaggle (/kaggle/input/...) and locally (../data)
kaggle_files = list(Path("/kaggle/input").rglob("train.csv")) if Path("/kaggle/input").exists() else []
DATA = kaggle_files[0].parent if kaggle_files else Path("../data")
OUTPUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
y = train[TARGET].to_numpy()  # día 1: codificar si hace falta (en S6E9, Yes/No -> 1/0)

print(f"train: {train.shape}   test: {test.shape}")
print("versions -> pandas", pd.__version__, "| lightgbm", lgb.__version__)
train.head()

### Chart style

Charts that may travel on their own (for example, to X) use one style: the **title states the conclusion**, a
subtitle says what is measured, guide lines point at what to look at, and a footer names the competition.

In [ ]:
DARK_GREY = "#333333"
MID_GREY = "#555555"
ACCENT = "#d62728"

WIDTH_PX, HEIGHT_PX, DPI = 1600, 900, 100


def shareable_figure(title, subtitle="", footer="", nrows=1, ncols=1, **subplots_kw):
    """Create a 1600x900 figure with a conclusion title, subtitle and footer."""
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(WIDTH_PX / DPI, HEIGHT_PX / DPI), dpi=DPI, **subplots_kw
    )
    fig.suptitle(title, x=0.02, y=0.97, ha="left", fontsize=22, fontweight="bold", color=DARK_GREY)
    if subtitle:
        fig.text(0.02, 0.90, subtitle, ha="left", fontsize=14, color=MID_GREY)
    if footer:
        fig.text(0.02, 0.02, footer, ha="left", fontsize=11, color=MID_GREY)
    for ax in fig.axes:
        style_axes(ax)
    fig.subplots_adjust(left=0.07, right=0.97, top=0.84, bottom=0.12)
    return fig, axes


def style_axes(ax):
    """Dark grey labels and ticks, no top/right spines."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(MID_GREY)
    ax.tick_params(colors=DARK_GREY, labelsize=12)
    ax.xaxis.label.set_color(DARK_GREY)
    ax.yaxis.label.set_color(DARK_GREY)
    ax.xaxis.label.set_size(13)
    ax.yaxis.label.set_size(13)


def annotate(ax, text, xy, xytext, color=ACCENT):
    """Callout with a guide line from the text to the point of interest."""
    ax.annotate(
        text,
        xy=xy,
        xytext=xytext,
        fontsize=12,
        color=color,
        arrowprops={"arrowstyle": "-", "color": color, "lw": 1.2},
    )


def guide_line(ax, x=None, y=None, label=None, color=ACCENT):
    """Dashed vertical (x) or horizontal (y) reference line, e.g. a base rate or a clip value."""
    if x is not None:
        ax.axvline(x, ls="--", lw=1.2, color=color)
        if label:
            ax.text(x, ax.get_ylim()[1], f" {label}", color=color, fontsize=11, va="top")
    if y is not None:
        ax.axhline(y, ls="--", lw=1.2, color=color)
        if label:
            ax.text(ax.get_xlim()[1], y, f"{label} ", color=color, fontsize=11, ha="right", va="bottom")


def save_shareable(fig, path, alt_text):
    """Save at exactly 1600x900 and write the alt text next to the image (same name, .txt)."""
    if not alt_text:
        raise ValueError("A chart published outside the notebook needs alt text.")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=DPI)
    path.with_suffix(".txt").write_text(alt_text.strip() + "\n", encoding="utf-8")
    return path


FOOTER = "Kaggle Playground S6E10 · <TODO: notebook title>"

## 1.2 Data dictionary: what each column means

<TODO>

In [ ]:
FEATURES = [c for c in test.columns if c != "id"]
CATEGORICAL = train[FEATURES].select_dtypes(exclude="number").columns.tolist()
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]

pd.DataFrame({
    "dtype": train[FEATURES].dtypes.astype(str),
    "n_unique": train[FEATURES].nunique(),
    "min": [train[c].min() for c in FEATURES],
    "max": [train[c].max() for c in FEATURES],
    "nulls": train[FEATURES].isna().sum(),
})

## 1.3 The target

<TODO: distribution; imbalance box if it applies>

In [ ]:
pd.Series(y).describe() if np.issubdtype(y.dtype, np.floating) else pd.Series(y).value_counts(normalize=True)

# Part 2 — The metric: <TODO>

<TODO: what the metric rewards, a naive baseline that shows its trap, practical consequences>

In [ ]:
from sklearn.metrics import roc_auc_score  # día 1: cambiar por la métrica real

def score(y_true, y_pred):
    return roc_auc_score(y_true, y_pred)  # día 1: métrica de la competición

HIGHER_IS_BETTER = True  # día 1: False si es un error (RMSE, logloss...)

# Part 3 — EDA: what drives the target?

<TODO>

In [ ]:
# día 1: tasa/valor medio del target por grupo, valores recortados, interacciones, train vs test

# Part 4 — Validation before modelling

This is the step most people skip, and the one that decides whether any later number means something.

> **Concept — overfitting.** A model *overfits* when it memorises the training data instead of learning patterns
> that generalise. Its score on the data it was trained on is then far better than on unseen data.

## 4.1 The folds

In [ ]:
# Stratified for classification; plain KFold for regression
splitter = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)  # día 1: KFold si es regresión
folds = list(splitter.split(train, y))
for k, (tr_idx, va_idx) in enumerate(folds):
    print(f"fold {k}: train {len(tr_idx):,} rows | validation {len(va_idx):,} rows")

All models in this notebook use **these exact folds**, so their scores are directly comparable.

## 4.2 The CV must track the leaderboard

The check that makes a CV *trustworthy*: **when CV goes up, the leaderboard should go up by a similar amount**.
If they disagree, fix the validation before tuning anything.

A small helper runs any model through the folds and reports per-fold score, mean ± std and OOF score:

In [ ]:
RESULTS = {}
FOLD_SCORES = {}


def run_cv(name, fit_predict, keep_first_fold=False):
    # fit_predict(tr_idx, va_idx) -> (validation predictions, test predictions, info string, extras)
    oof, test_pred, scores = np.zeros(len(train)), np.zeros(len(test)), []
    start, first_fold = time.time(), None
    for k, (tr_idx, va_idx) in enumerate(folds):
        va_pred, te_pred, info, extras = fit_predict(tr_idx, va_idx)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        scores.append(score(y[va_idx], va_pred))
        if keep_first_fold and k == 0:
            first_fold = extras
        print(f"  fold {k}: {scores[-1]:.5f} {info}")
    RESULTS[name] = {"cv_mean": np.mean(scores), "cv_std": np.std(scores), "oof": score(y, oof)}
    FOLD_SCORES[name] = scores
    print(f"{name}: CV {np.mean(scores):.5f} ± {np.std(scores):.5f} | OOF {RESULTS[name]['oof']:.5f} "
          f"| {time.time() - start:.0f}s")
    return test_pred, first_fold

# Part 5 — Model 1: <TODO: linear baseline>

<TODO>

In [ ]:
# día 1: baseline lineal (en S6E9, StandardScaler + OneHotEncoder + LogisticRegression)

# Part 6 — Model 2: gradient-boosted trees with LightGBM

<TODO>

In [ ]:
for col in CATEGORICAL:
    levels = sorted(set(train[col]) | set(test[col]))  # same codes in train and test
    train[col] = pd.Categorical(train[col], categories=levels)
    test[col] = pd.Categorical(test[col], categories=levels)

LGB_PARAMS = {
    "objective": "binary",  # día 1: regression / multiclass según el problema
    "metric": "auc",        # día 1: la métrica de la competición
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "seed": SEED,
    "verbose": -1,
}


def make_lgbm(get_frames):
    # get_frames(tr_idx, va_idx) -> (X_tr, X_va, X_te) with the columns to use
    def fit_predict(tr_idx, va_idx):
        X_tr, X_va, X_te = get_frames(tr_idx, va_idx)
        booster = lgb.train(
            LGB_PARAMS,
            lgb.Dataset(X_tr, y[tr_idx]),
            num_boost_round=5000,
            valid_sets=[lgb.Dataset(X_va, y[va_idx])],
            callbacks=[lgb.early_stopping(200, verbose=False)],
        )
        return (booster.predict(X_va), booster.predict(X_te),
                f"(trees: {booster.best_iteration})", (booster, X_va))
    return fit_predict


def raw_frames(tr_idx, va_idx):
    return train[FEATURES].iloc[tr_idx], train[FEATURES].iloc[va_idx], test[FEATURES]


test_pred_raw, (raw_booster, raw_X_va) = run_cv("LightGBM (raw features)", make_lgbm(raw_frames), keep_first_fold=True)

# Part 7 — Feature engineering

<TODO: the idea that comes from understanding the data (in S6E9, the values copied from the original dataset)>

In [ ]:
# día 2+: features, siempre fold-safe

# Part 8 — Interpreting the model with SHAP

> **Concept — SHAP values.** For one prediction, SHAP splits the difference between that prediction and the average
> prediction into a **contribution per feature**. For LightGBM they are computed exactly and they are **additive**:
> average + sum of contributions = the model's raw output for that row.

LightGBM computes them natively with `pred_contrib=True` — no extra library needed.

In [ ]:
final_booster, final_X_va = raw_booster, raw_X_va  # día 2+: el mejor modelo
sample = final_X_va.sample(min(20_000, len(final_X_va)), random_state=SEED)
contrib = final_booster.predict(sample, pred_contrib=True)
shap_values = pd.DataFrame(contrib[:, :-1], columns=sample.columns, index=sample.index)
expected_value = contrib[0, -1]

raw = final_booster.predict(sample, raw_score=True)
print("max |base + sum(SHAP) - raw score| =", float(np.abs(expected_value + shap_values.sum(axis=1) - raw).max()))
shap_values.abs().mean().sort_values(ascending=False)

# Part 9 — What helped and what did not

An honest ablation: each row changes one thing, all with the same folds.

| Model | CV | Public LB |
|---|---|---|
| <TODO> | | |

# Part 10 — Summary

<TODO>

## Glossary

| Term | In one line |
|---|---|
| <TODO> | |

In [ ]:
test_pred_final = test_pred_raw  # día 2+: el mejor modelo
submission = pd.DataFrame({"id": test["id"], TARGET: test_pred_final})
submission.to_csv(OUTPUT / "submission.csv", index=False)
submission.head()

## References

- Competition: [Playground Series S6E10](https://www.kaggle.com/competitions/playground-series-s6e10)
- Original dataset: <TODO>
- <TODO: credit whoever inspired an idea>

## Acknowledgements

This notebook was written with the assistance of Claude (Anthropic), an AI model. Analysis direction, review and
final decisions are the author's.